# DD2360 Assignment 2
Runtime → Change runtime type → **T4 GPU** before running. All build output goes to `build/`.

In [4]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
REPO = "tattack99/DD2360-assignment-2"
REPO_DIR = "/content/repo"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        from google.colab import userdata
        token = userdata.get("GH_TOKEN_DD2360_ASSIGNMENT_2")
        !git clone -q https://{token}@github.com/{REPO}.git {REPO_DIR}
    %cd {REPO_DIR}
    !git pull -q

/content/repo


In [5]:
# Check that a GPU is attached and nvcc is available
!nvidia-smi
!nvcc --version

Wed Nov 18 07:42:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Q1 - 1D Convolution

In [6]:
# Quick check: both kernels should print PASS
!make run

nvcc -arch=native -DTILE_WIDTH=256 -o build/hw2-convolution_tile256 assignment/hw2-convolution.cu
./build/hw2-convolution_tile256 1024
Timing - Allocated host memory                                          4 us
Timing - Allocated device memory                                   321787 us
Timing - Initialize arrays with values                                 28 us
Timing - Copying data to the GPU                                      527 us
Timing - Finished 1D convolution(basic)                               608 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         47 us
Timing - Finished 1D convolution(tiled)                                34 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         46 us
Basic kernel vs CPU: PASS (0 mismatches)
Tiled kernel vs CP

In [7]:
# Q1.3 tile size: run every tile size on the largest input, compare the tiled kernel times
!make tilesweep ARGS=16384

=== TILE = 32 ===
nvcc -arch=native -DTILE_WIDTH=32 -o build/hw2-convolution_tile32 assignment/hw2-convolution.cu
./build/hw2-convolution_tile32 16384
Timing - Allocated host memory                                          9 us
Timing - Allocated device memory                                   272557 us
Timing - Initialize arrays with values                                398 us
Timing - Copying data to the GPU                                       73 us
Timing - Finished 1D convolution(basic)                               122 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         69 us
Timing - Finished 1D convolution(tiled)                                56 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         93 us
Basic kernel vs CPU: PASS (0 mismatches)
Ti

In [8]:
# Q1.3 screenshot: N = 1024 ... 16384 with the chosen tile size
TILE = 256
!make sweep TILE={TILE}

=== N = 1024, TILE = 256 ===
Timing - Allocated host memory                                          5 us
Timing - Allocated device memory                                   275448 us
Timing - Initialize arrays with values                                 32 us
Timing - Copying data to the GPU                                       47 us
Timing - Finished 1D convolution(basic)                               136 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         36 us
Timing - Finished 1D convolution(tiled)                                36 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         34 us
Basic kernel vs CPU: PASS (0 mismatches)
Tiled kernel vs CPU: PASS (0 mismatches)
Timing - Free memory resources                                        122 us
=== N 

In [9]:
# Q1.4 Nsight Systems: where the time goes
!make profile TILE={TILE} ARGS=16384

nsys profile --stats=true --force-overwrite true -o build/hw2-convolution-report ./build/hw2-convolution_tile256 16384
Timing - Allocated host memory                                         12 us
Timing - Allocated device memory                                   511772 us
Timing - Initialize arrays with values                                469 us
Timing - Copying data to the GPU                                       83 us
Timing - Finished 1D convolution(basic)                               843 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         67 us
Timing - Finished 1D convolution(tiled)                                80 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         85 us
Basic kernel vs CPU: PASS (0 mismatches)
Tiled kernel vs CPU: PASS (0 misma

In [10]:
# Q1.4 Nsight Compute: occupancy, shared memory, memory throughput
!make ncu TILE={TILE} ARGS=16384

ncu --section SpeedOfLight --section MemoryWorkloadAnalysis --section Occupancy --section LaunchStats ./build/hw2-convolution_tile256 16384
Timing - Allocated host memory                                          6 us
==PROF== Connected to process 10156 (/content/repo/build/hw2-convolution_tile256)
Timing - Allocated device memory                                   645668 us
Timing - Initialize arrays with values                                471 us
Timing - Copying data to the GPU                                       82 us
==PROF== Profiling "convolution_1D_basic" - 0: 0%....50%....100% - 9 passes
Timing - Finished 1D convolution(basic)                           3343172 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results        135 us
==PROF== Profiling "convolution_1D_tiled" - 1: 0%....50%....100% - 9 passes
Timing - Finished 1D convolution(tiled)                           1856602 us
P_tiled